In [2]:
import os
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta
from dotenv import load_dotenv
from opendartreader import OpenDartReader

# API 키 로드 및 DART 객체 생성
load_dotenv()
api_key = os.environ.get('DART_API_KEY')
dart = OpenDartReader(api_key)

# 매핑 테이블 불러오기
current_path = os.getcwd() # 현재 위치: .../investment-risk-detector/notebooks
root_path = os.path.dirname(current_path) # 한 단계 위(프로젝트 루트)로 이동: .../investment-risk-detector
csv_path = os.path.join(root_path, 'data', 'kospi_top50_mapping.csv') # 루트/data/... 로 경로 재지정

print(f"✅ 파일을 불러올 경로: {csv_path}")

# CSV에서 숫자를 읽어올 때 앞의 '0'이 날아갈 수 있으므로 문자열(str)
mapping_df = pd.read_csv(csv_path, dtype={'corp_code': str}) 

# 삼성전자 고유번호 추출 및 8자리 포맷팅(zfill)
raw_code = mapping_df.loc[mapping_df['corp_name'] == '삼성전자', 'corp_code'].values[0]
samsung_corp_code = str(raw_code).zfill(8) # '126380' -> '00126380' 변환
print(f"✅ 삼성전자 DART 고유번호: {samsung_corp_code}")

# 조회 기간 설정 (오늘 기준 1년 전)
today = datetime.now()
one_year_ago = today - relativedelta(years=1)

# API가 요구하는 'YYYYMMDD' 포맷으로 변환
bgn_de = one_year_ago.strftime('%Y%m%d')
end_de = today.strftime('%Y%m%d')
print(f"✅ 조회 기간: {bgn_de} ~ {end_de}")

# OpenDART API로 공시 목록 조회
samsung_disclosures = dart.list(samsung_corp_code, start=bgn_de, end=end_de)

# 결과 확인
print(f"✅ 수집된 공시 개수: {len(samsung_disclosures)}건")
display(samsung_disclosures.head(15))

✅ 파일을 불러올 경로: C:\Users\YUN\Project\investment-risk-detector\data\kospi_top50_mapping.csv
✅ 삼성전자 DART 고유번호: 00126380
✅ 조회 기간: 20250827 ~ 20260827
✅ 수집된 공시 개수: 2847건


,corp_code,corp_name,stock_code,corp_cls,report_nm,rcept_no,flr_nm,rcept_dt,rm
0,00126380,삼성전자,005930,Y,임원ㆍ주요주주특정증권등소유상황보고서,20260826000445,김경륜,20260826,
1,00126380,삼성전자,005930,Y,[기재정정]임원ㆍ주요주주특정증권등소유상황보고서,20260826000434,조미선,20260826,
2,00126380,삼성전자,005930,Y,임원ㆍ주요주주특정증권등소유상황보고서,20260824000222,안성준,20260824,
3,00126380,삼성전자,005930,Y,기타경영사항(자율공시),20260821800837,삼성전자,20260821,유
4,00126380,삼성전자,005930,Y,주요사항보고서(자기주식취득결정),20260821000616,삼성전자,20260821,
5,00126380,삼성전자,005930,Y,수시공시의무관련사항(공정공시),20260821800763,삼성전자,20260821,유
6,00126380,삼성전자,005930,Y,임원ㆍ주요주주특정증권등소유상황보고서,20260821000262,이명재,20260821,
7,00126380,삼성전자,005930,Y,임원ㆍ주요주주특정증권등소유상황보고서,20260820000290,김경태,20260820,
8,00126380,삼성전자,005930,Y,임원ㆍ주요주주특정증권등소유상황보고서,20260820000040,이희곤,20260820,
9,00126380,삼성전자,005930,Y,[기재정정]임원ㆍ주요주주특정증권등소유상황보고서,20260814003973,박태훈,20260814,


In [19]:
# 제출인명(flr_nm)이 법인명('삼성전자')과 일치하는 데이터만 필터링
corp_name = '삼성전자'
filtered_df = samsung_disclosures[samsung_disclosures['flr_nm'] == corp_name].copy()

# 필터링 전후 데이터 건수 비교
original_count = len(samsung_disclosures)
filtered_count = len(filtered_df)
print(f"✅ 전체 공시 건수: {original_count}건")
print(f"✅ 법인 제출 공시 건수: {filtered_count}건 (노이즈 {original_count - filtered_count}건 제거됨!)")

# 결과 확인
display(filtered_df[['rcept_dt', 'report_nm', 'flr_nm']].head(15))

✅ 전체 공시 건수: 2847건
✅ 법인 제출 공시 건수: 120건 (노이즈 2727건 제거됨!)


,rcept_dt,report_nm,flr_nm
3,20260821,기타경영사항(자율공시),삼성전자
4,20260821,주요사항보고서(자기주식취득결정),삼성전자
5,20260821,수시공시의무관련사항(공정공시),삼성전자
12,20260814,반기보고서 (2026.06),삼성전자
14,20260814,동일인등출자계열회사와의상품ㆍ용역거래변경,삼성전자
15,20260814,동일인등출자계열회사와의상품ㆍ용역거래변경,삼성전자
16,20260814,동일인등출자계열회사와의상품ㆍ용역거래변경,삼성전자
17,20260814,지급수단별ㆍ지급기간별지급금액및분쟁조정기구에관한사항,삼성전자
43,20260731,최대주주등소유주식변동신고서,삼성전자
47,20260730,특수관계인과의보험거래,삼성전자


In [20]:
# 리스크 탐지 키워드 리스트 정의
risk_keywords = [
    '유상증자', '감자', '소송', '횡령', '배임', 
    '해지', '생산중단', '영업정지', '전환사채', '신주인수권부사채'
]

# 리스트를 정규표현식 OR(|) 패턴으로 변환 (예: '유상증자|감자|소송|...')
pattern = '|'.join(risk_keywords)
print(f"✅ 탐지 패턴: {pattern}")

# report_nm(보고서명)에 키워드가 포함된 공시만 추출
risk_df = filtered_df[filtered_df['report_nm'].str.contains(pattern, regex=True, na=False)].copy()

# 결과 확인
print(f"✅ 1년간 발견된 잠재적 위험 공시 건수: {len(risk_df)}건")
display(risk_df[['rcept_dt', 'report_nm', 'rcept_no']])

✅ 탐지 패턴: 유상증자|감자|소송|횡령|배임|해지|생산중단|영업정지|전환사채|신주인수권부사채
✅ 1년간 발견된 잠재적 위험 공시 건수: 0건


,rcept_dt,report_nm,rcept_no
